### Procesamiento de Lenguaje Natural I
# **Desafío 1**



In [26]:
%pip install numpy scikit-learn

Note: you may need to restart the kernel to use updated packages.


### Vectorización de texto y modelo de clasificación Naïve Bayes con el dataset 20 newsgroups

In [1]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.metrics import f1_score

Utilizamos **20newsgroups** por ser un dataset clásico de NLP ya viene incluido y formateado en sklearn

In [2]:
from sklearn.datasets import fetch_20newsgroups
import numpy as np

## Carga de datos

Cargamos los datos (ya separados de forma predeterminada en train y test)

El dataset 20 Newsgroups contiene aproximadamente 18 000 publicaciones de grupos de noticias distribuidas en 20 temas. Está dividido en dos subconjuntos: uno para entrenamiento (train set) y otro para pruebas (test set).

In [3]:
newsgroups_train = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))
newsgroups_test = fetch_20newsgroups(subset='test', remove=('headers', 'footers', 'quotes'))

## Vectorización

Instanciamos un vectorizador.

Podemos ver diferentes parámetros de instanciación en la documentación de sklearn https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html

In [4]:
tfidfvect = TfidfVectorizer()

En el atributo `data` accedemos al texto

In [31]:
print(newsgroups_train.data[0])

I was wondering if anyone out there could enlighten me on this car I saw
the other day. It was a 2-door sports car, looked to be from the late 60s/
early 70s. It was called a Bricklin. The doors were really small. In addition,
the front bumper was separate from the rest of the body. This is 
all I know. If anyone can tellme a model name, engine specs, years
of production, where this car is made, history, or whatever info you
have on this funky looking car, please e-mail.


Con la interfaz habitual de sklearn podemos ajustar el vectorizador (obtener el vocabulario y calcular el vector IDF) y transformar directamente los datos.

Podemos denominar `X_train` como la matriz documento-término.

In [5]:
X_train = tfidfvect.fit_transform(newsgroups_train.data)

Recordemos que las vectorizaciones por conteos son de tipo sparse, por ello sklearn convenientemente devuelve los vectores de documentos como matrices de tipo sparse.

In [33]:
print(type(X_train))
print(f'shape: {X_train.shape}')
print(f'Cantidad de documentos: {X_train.shape[0]}')
print(f'Tamaño del vocabulario (dimensionalidad de los vectores): {X_train.shape[1]}')

<class 'scipy.sparse._csr.csr_matrix'>
shape: (11314, 101631)
Cantidad de documentos: 11314
Tamaño del vocabulario (dimensionalidad de los vectores): 101631


Una vez ajustado el vectorizador, podemos acceder a atributos como el vocabulario aprendido. Es un diccionario que va de términos a índices.

El índice es la posición en el vector de documento.

In [34]:
tfidfvect.vocabulary_['car']

25775

Probamos con una palabra que no está en el documento.

In [35]:
tfidfvect.vocabulary_['cocoliso']

KeyError: 'cocoliso'

Es muy útil tener el diccionario opuesto que va de índices a términos

In [6]:
idx2word = {v: k for k,v in tfidfvect.vocabulary_.items()}

En `y_train` guardamos los targets que son enteros

In [7]:
y_train = newsgroups_train.target
y_train[:10]

array([ 7,  4,  4,  1, 14, 16, 13,  3,  2,  4])

Hay 20 clases correspondientes a los 20 grupos de noticias

In [ ]:
print(f'clases {np.unique(newsgroups_test.target)}')
newsgroups_test.target_names

clases [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19]


['alt.atheism',
 'comp.graphics',
 'comp.os.ms-windows.misc',
 'comp.sys.ibm.pc.hardware',
 'comp.sys.mac.hardware',
 'comp.windows.x',
 'misc.forsale',
 'rec.autos',
 'rec.motorcycles',
 'rec.sport.baseball',
 'rec.sport.hockey',
 'sci.crypt',
 'sci.electronics',
 'sci.med',
 'sci.space',
 'soc.religion.christian',
 'talk.politics.guns',
 'talk.politics.mideast',
 'talk.politics.misc',
 'talk.religion.misc']

## Similaridad de documentos

Veamos similaridad de documentos. Tomemos algún documento

In [ ]:
idx = 4811
print(newsgroups_train.data[idx])

THE WHITE HOUSE

                  Office of the Press Secretary
                   (Pittsburgh, Pennslyvania)
______________________________________________________________
For Immediate Release                         April 17, 1993     

             
                  RADIO ADDRESS TO THE NATION 
                        BY THE PRESIDENT
             
                Pittsburgh International Airport
                    Pittsburgh, Pennsylvania
             
             
10:06 A.M. EDT
             
             
             THE PRESIDENT:  Good morning.  My voice is coming to
you this morning through the facilities of the oldest radio
station in America, KDKA in Pittsburgh.  I'm visiting the city to
meet personally with citizens here to discuss my plans for jobs,
health care and the economy.  But I wanted first to do my weekly
broadcast with the American people. 
             
             I'm told this station first broadcast in 1920 when
it reported that year's presidential elec

Medimos la similaridad coseno con todos los documentos de train

In [ ]:
cossim = cosine_similarity(X_train[idx], X_train)[0]

Podemos ver los valores de similaridad ordenados de mayor a menor

In [ ]:
np.sort(cossim)[::-1]

array([1.        , 0.70930477, 0.67474953, ..., 0.        , 0.        ,
       0.        ])

Después vemos a qué documentos corresponden

In [ ]:
np.argsort(cossim)[::-1]

array([4811, 6635, 4253, ..., 1911, 1825, 1828])

Obtenemos los 5 documentos más similares:

In [ ]:
mostsim = np.argsort(cossim)[::-1][1:6]
print(mostsim)

[6635 4253 3596 4271 3746]


El documento original pertenece a la clase:

In [ ]:
newsgroups_train.target_names[y_train[idx]]

'talk.politics.misc'

Revisamos las clases de los 5 más similares:

In [ ]:
for i in mostsim:
  print(newsgroups_train.target_names[y_train[i]])

talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc


### Modelo de clasificación Naïve Bayes

Instanciamos el modelo de clasificación Naive Bayes y lo entrenamos con sklearn

In [ ]:
clf = MultinomialNB()
clf.fit(X_train, y_train)

,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


Ya tenemos nuestro vectorizador ya ajustado en train, vectorizamos los textos
del conjunto de test.

In [ ]:
X_test = tfidfvect.transform(newsgroups_test.data)
y_test = newsgroups_test.target
y_pred =  clf.predict(X_test)

El F1-score es una métrica adecuada para evaluar el desempeño de modelos de clasificación, especialmente cuando existe desbalance entre clases.

* El promediado macro calcula el promedio del F1-score de cada clase, otorgando el mismo peso a todas las clases.
* El promediado micro calcula las métricas de forma global considerando todas las predicciones; en problemas de clasificación multiclase suele ser equivalente a la accuracy, por lo que no es la mejor métrica cuando el dataset está desbalanceado.

In [ ]:
f1_score(y_test, y_pred, average='macro')

0.5854345727938506

---

## **Consigna del Desafío 1**
**Cada experimento realizado debe estar acompañado de una explicación o interpretación de lo observado.**



**1. Vectorizar documentos**
* Tomar 5 documentos al azar y medir similaridad con el resto de los documentos.
Estudiar los 5 documentos más similares de cada uno analizar si tiene sentido
la similaridad según el contenido del texto y la etiqueta de clasificación.

**2. Construir un modelo de clasificación por prototipos (tipo zero-shot).**
* Clasificar los documentos de un conjunto de test comparando cada uno con todos los de entrenamiento y asignar la clase al label del documento del conjunto de entrenamiento con mayor similaridad.

**3. Entrenar modelos de clasificación Naïve Bayes para maximizar el desempeño de clasificación**

* F1-Score Macro en el conjunto de datos de test. Considerar cambiar parámetros
de instanciación del vectorizador y los modelos y probar modelos de Naïve Bayes Multinomial y ComplementNB.

**NO cambiar el hiperparámetro ngram_range de los vectorizadores**.

**4. Transponer la matriz documento-término.**
* De esa manera se obtiene una matriz término-documento que puede ser interpretada como una colección de vectorización de palabras.
* Estudiar ahora similaridad entre palabras tomando 5 palabras y estudiando sus 5 más similares.

**Elegir las palabras MANUALMENTE para evitar la aparición de términos poco interpretables**.


In [12]:

# ============================================================
# 1. Vectorizar documentos y medir similaridad entre ellos
# ============================================================
np.random.seed(42)
random_doc_indices = np.random.choice(X_train.shape[0], 5, replace=False)

for idx in random_doc_indices:
    cossim = cosine_similarity(X_train[idx], X_train)[0]
    # Los 5 más similares (excluimos el propio documento, índice 0)
    top5 = np.argsort(cossim)[::-1][1:6]

    doc_class = newsgroups_train.target_names[y_train[idx]]
    print(f"\n{'='*70}")
    print(f"Documento #{idx} | Clase: '{doc_class}'")
    print(f"Texto (primeros 300 chars):\n{newsgroups_train.data[idx][:300]}")
    print(f"\nTop 5 documentos más similares:")
    for rank, sim_idx in enumerate(top5, 1):
        sim_class = newsgroups_train.target_names[y_train[sim_idx]]
        match = "✓ misma clase" if sim_class == doc_class else "✗ clase diferente"
        print(f"  {rank}. Doc #{sim_idx} | Clase: '{sim_class}' [{match}] | Sim: {cossim[sim_idx]:.4f}")



Documento #7492 | Clase: 'comp.sys.mac.hardware'
Texto (primeros 300 chars):
Could someone please post any info on these systems.

Thanks.
BoB
-- 
---------------------------------------------------------------------- 
Robert Novitskey | "Pursuing women is similar to banging one's head
rrn@po.cwru.edu  |  against a wall...with less opportunity for reward" 

Top 5 documentos más similares:
  1. Doc #10935 | Clase: 'comp.sys.mac.hardware' [✓ misma clase] | Sim: 0.6665
  2. Doc #7258 | Clase: 'comp.sys.ibm.pc.hardware' [✗ clase diferente] | Sim: 0.3476
  3. Doc #4971 | Clase: 'comp.sys.mac.hardware' [✓ misma clase] | Sim: 0.1799
  4. Doc #4303 | Clase: 'misc.forsale' [✗ clase diferente] | Sim: 0.1547
  5. Doc #645 | Clase: 'comp.sys.mac.hardware' [✓ misma clase] | Sim: 0.1414

Documento #3546 | Clase: 'comp.os.ms-windows.misc'
Texto (primeros 300 chars):


     Don't bother if you have CPBackup or Fastback.  They all offer options 
not available in the stripped-down MS version (FROM CPS!


**Interpretación (Ejercicio 1):**

Los resultados son bastante variados entre los 5 documentos seleccionados.

| Doc # | Clase | Coincidencias (de 5) | Sim. máxima |
|-------|-------|:---:|:---:|
| #7492 | `comp.sys.mac.hardware` | 3/5 | 0.67 |
| #3546 | `comp.os.ms-windows.misc` | 0/5 | 0.20 |
| #5582 | `misc.forsale` | 3/5 | 0.46 |
| #4793 | `talk.politics.guns` | 2/5 | 0.24 |
| #3813 | `rec.sport.hockey` | 0/5 | 0.25 |

El doc #7492 tiene un vecino muy cercano (sim. 0.67) y 3 de 5 son de la misma clase. Los otros dos son de `comp.sys.ibm.pc.hardware`, que comparte mucho vocabulario de hardware. Similar para #5582 (misc.forsale): 3 coinciden, y los dos de `comp.graphics` aparecen porque varios posts de esa categoría venden hardware gráfico.

Los casos que fallan son más interesantes. El #3546 (`comp.os.ms-windows.misc`) no encuentra ningún vecino de su clase —los 5 son de `comp.sys.ibm.pc.hardware`—, lo que sugiere que el post trata algún problema de drivers o configuración de hardware, vocabulario que cruza esa frontera fácilmente. El #4793 (`talk.politics.guns`) solo tiene 2 coincidencias; entre los vecinos aparecen `sci.crypt` y `talk.politics.misc`, categorías que se superponen en discusiones sobre libertades civiles.

El caso más llamativo es el #3813 (`rec.sport.hockey`): ninguno de los 5 vecinos es de su clase, todos son de `alt.atheism` o `soc.religion.christian`. Lo más probable es que el post en cuestión sea una discusión filosófica o moral —algo no infrecuente en los grupos de noticias de la época— y no hable de hockey en absoluto. TF-IDF no tiene forma de saberlo.

En general, cuando la similitud máxima supera 0.4 los vecinos tienden a coincidir en clase. Por debajo de 0.25 el documento suele tener vocabulario ambiguo o compartido con varias categorías a la vez.



---
## 2. Clasificación por Prototipos (Zero-Shot)

Para cada documento de **test**, calculamos su similaridad coseno con **todos** los documentos de **train** y asignamos la clase del documento de entrenamiento más similar.  
Procesamos en lotes (`batch_size=500`) para no cargar toda la matriz de similaridad en memoria de una vez.


In [9]:

# ============================================================
# 2. Clasificación por prototipos (zero-shot)
# ============================================================
# Vectorizamos test con el mismo vectorizador ya ajustado en train
X_test = tfidfvect.transform(newsgroups_test.data)
y_test = newsgroups_test.target

batch_size = 500
n_test = X_test.shape[0]
most_similar_idx = np.zeros(n_test, dtype=int)

for start in range(0, n_test, batch_size):
    end = min(start + batch_size, n_test)
    sim_batch = cosine_similarity(X_test[start:end], X_train)
    most_similar_idx[start:end] = np.argmax(sim_batch, axis=1)

y_pred_proto = y_train[most_similar_idx]
f1_proto = f1_score(y_test, y_pred_proto, average='macro')
print(f"F1-Score Macro (clasificación por prototipos): {f1_proto:.4f}")


F1-Score Macro (clasificación por prototipos): 0.5050



**Interpretación (Ejercicio 2):**

El clasificador por prototipos obtuvo **F1-Macro = 0.5050**. La idea es simple: para cada documento de test se busca el más parecido en train y se le asigna su etiqueta —básicamente un 1-NN en el espacio TF-IDF, sin entrenamiento explícito.

El resultado queda bastante por debajo del Naïve Bayes (0.59 baseline, 0.69 con el mejor modelo). Tiene sentido: NB aprovecha la distribución estadística de *todas* las palabras a lo largo de *toda* la clase, mientras que este enfoque depende de un solo documento. Si ese vecino más cercano es un post ruidoso o atípico, la clasificación falla aunque el documento de test sea perfectamente representativo de su categoría.

Dicho eso, un F1 de 0.50 sobre 20 clases no es malo para algo completamente zero-shot. Confirma que los vectores TF-IDF agrupan documentos temáticamente, aunque no con la precisión suficiente para competir con un modelo entrenado.



---
## 3. Optimización de modelos Naïve Bayes

Se prueban distintas combinaciones de parámetros del vectorizador (`TfidfVectorizer` / `CountVectorizer`) junto con `MultinomialNB` y `ComplementNB`.  
**Restricción:** no se modifica `ngram_range`.


In [10]:

# ============================================================
# 3. Entrenar modelos Naïve Bayes para maximizar F1-Score Macro
# ============================================================
from sklearn.feature_extraction.text import CountVectorizer

resultados = []

# --- Baseline: TF-IDF defaults + MultinomialNB ---
vect_base = TfidfVectorizer()
Xtr_base = vect_base.fit_transform(newsgroups_train.data)
Xte_base = vect_base.transform(newsgroups_test.data)
clf_base = MultinomialNB()
clf_base.fit(Xtr_base, y_train)
y_pred_base = clf_base.predict(Xte_base)
resultados.append(("Baseline: TF-IDF defaults + MultinomialNB",
                   f1_score(y_test, y_pred_base, average='macro')))

# --- Experimento 1: sublinear_tf + min_df + MultinomialNB ---
vect1 = TfidfVectorizer(sublinear_tf=True, min_df=2, max_df=0.95)
Xtr1 = vect1.fit_transform(newsgroups_train.data)
Xte1 = vect1.transform(newsgroups_test.data)
m1 = MultinomialNB(alpha=0.1)
m1.fit(Xtr1, y_train)
resultados.append(("TF-IDF(sublinear, min_df=2) + MultinomialNB(α=0.1)",
                   f1_score(y_test, m1.predict(Xte1), average='macro')))

# --- Experimento 2: mismo vectorizador + ComplementNB ---
m2 = ComplementNB(alpha=0.1)
m2.fit(Xtr1, y_train)
resultados.append(("TF-IDF(sublinear, min_df=2) + ComplementNB(α=0.1)",
                   f1_score(y_test, m2.predict(Xte1), average='macro')))

# --- Experimento 3: más filtros + ComplementNB alpha pequeño ---
vect3 = TfidfVectorizer(sublinear_tf=True, min_df=3, max_df=0.9, max_features=50000)
Xtr3 = vect3.fit_transform(newsgroups_train.data)
Xte3 = vect3.transform(newsgroups_test.data)
m3 = ComplementNB(alpha=0.05)
m3.fit(Xtr3, y_train)
resultados.append(("TF-IDF(sublinear, min_df=3, max_feat=50k) + ComplementNB(α=0.05)",
                   f1_score(y_test, m3.predict(Xte3), average='macro')))

# --- Experimento 4: CountVectorizer + ComplementNB ---
vect4 = CountVectorizer(min_df=2, max_df=0.95)
Xtr4 = vect4.fit_transform(newsgroups_train.data)
Xte4 = vect4.transform(newsgroups_test.data)
m4 = ComplementNB(alpha=0.1)
m4.fit(Xtr4, y_train)
resultados.append(("CountVectorizer(min_df=2) + ComplementNB(α=0.1)",
                   f1_score(y_test, m4.predict(Xte4), average='macro')))

# Imprimir resultados ordenados de mejor a peor
print(f"{'Configuración':<58} {'F1-Macro':>8}")
print("-" * 68)
for nombre, f1 in sorted(resultados, key=lambda x: x[1], reverse=True):
    print(f"{nombre:<58} {f1:>8.4f}")


Configuración                                              F1-Macro
--------------------------------------------------------------------
TF-IDF(sublinear, min_df=2) + ComplementNB(α=0.1)            0.6900
TF-IDF(sublinear, min_df=3, max_feat=50k) + ComplementNB(α=0.05)   0.6816
TF-IDF(sublinear, min_df=2) + MultinomialNB(α=0.1)           0.6716
CountVectorizer(min_df=2) + ComplementNB(α=0.1)              0.6385
Baseline: TF-IDF defaults + MultinomialNB                    0.5854



**Interpretación (Ejercicio 3):**

| Configuración | F1-Macro |
|---|:---:|
| TF-IDF(sublinear, min_df=2) + **ComplementNB**(α=0.1) | **0.6900** |
| TF-IDF(sublinear, min_df=3, max_feat=50k) + ComplementNB(α=0.05) | 0.6816 |
| TF-IDF(sublinear, min_df=2) + MultinomialNB(α=0.1) | 0.6716 |
| CountVectorizer(min_df=2) + ComplementNB(α=0.1) | 0.6385 |
| Baseline: TF-IDF defaults + MultinomialNB | 0.5854 |

La diferencia más visible es entre los dos clasificadores: ComplementNB supera a MultinomialNB en todas las configuraciones equivalentes. ComplementNB estima la distribución de cada clase modelando el "resto" del corpus en lugar de la clase en sí, lo que le da estimaciones más robustas cuando hay superposición de vocabulario entre categorías —algo bastante frecuente en 20 Newsgroups, donde categorías como `sci.med` y `sci.space` comparten terminología técnica.

Activar `sublinear_tf` fue el cambio más impactante: solo con eso y `min_df=2` se pasa de 0.5854 a 0.6716 en MultinomialNB. La transformación `1 + log(tf)` evita que una palabra que aparece 50 veces en un documento pese 50 veces más que una que aparece una sola vez, algo que distorsiona bastante la representación en textos con repeticiones. `min_df=2` descarta palabras que aparecen en un único documento —que no aportan poder discriminativo, solo dimensionalidad.

Restringir más con `min_df=3` y `max_features=50000` baja ligeramente el F1 (0.6816 vs 0.6900), lo que sugiere que esos términos con frecuencia exactamente 2 todavía aportan algo de señal.

La versión con CountVectorizer queda por debajo de todas las variantes TF-IDF. Sin la ponderación IDF, los términos comunes a todo el corpus dominan la representación y la hacen menos discriminativa entre clases.



---
## 4. Similaridad entre Palabras (Matriz Término-Documento)

Transponemos la matriz documento-término para obtener una **matriz término-documento** donde cada fila es un vector de palabra (su distribución sobre los documentos).  
Elegimos **5 palabras manualmente** con significado temático claro dentro del dataset 20 Newsgroups.


In [11]:

# ============================================================
# 4. Similaridad entre palabras (matriz término-documento)
# ============================================================
# Transponemos X_train: shape (vocab_size, n_docs)
X_term_doc = X_train.T

# Palabras elegidas manualmente: temáticas del dataset 20 Newsgroups
# (evitamos stopwords y términos técnicos poco interpretables)
palabras = ['computer', 'religion', 'space', 'gun', 'medical']

for word in palabras:
    if word not in tfidfvect.vocabulary_:
        print(f"'{word}' no está en el vocabulario.")
        continue

    word_idx = tfidfvect.vocabulary_[word]
    word_vec = X_term_doc[word_idx]          # vector fila de esa palabra
    word_sims = cosine_similarity(word_vec, X_term_doc)[0]

    # Los 5 más similares (excluimos la propia palabra)
    top5 = np.argsort(word_sims)[::-1][1:6]

    print(f"\nPalabra: '{word}'  →  Top 5 más similares:")
    for rank, i in enumerate(top5, 1):
        print(f"  {rank}. '{idx2word[i]}'  (similaridad: {word_sims[i]:.4f})")



Palabra: 'computer'  →  Top 5 más similares:
  1. 'decwriter'  (similaridad: 0.1563)
  2. 'deluged'  (similaridad: 0.1522)
  3. 'harkens'  (similaridad: 0.1522)
  4. 'shopper'  (similaridad: 0.1443)
  5. 'the'  (similaridad: 0.1361)

Palabra: 'religion'  →  Top 5 más similares:
  1. 'religious'  (similaridad: 0.2451)
  2. 'religions'  (similaridad: 0.2116)
  3. 'categorized'  (similaridad: 0.2039)
  4. 'purpsoe'  (similaridad: 0.2008)
  5. 'crusades'  (similaridad: 0.1987)

Palabra: 'space'  →  Top 5 más similares:
  1. 'nasa'  (similaridad: 0.3304)
  2. 'seds'  (similaridad: 0.2966)
  3. 'shuttle'  (similaridad: 0.2928)
  4. 'enfant'  (similaridad: 0.2803)
  5. 'seti'  (similaridad: 0.2465)

Palabra: 'gun'  →  Top 5 más similares:
  1. 'guns'  (similaridad: 0.3582)
  2. 'crime'  (similaridad: 0.2441)
  3. 'handgun'  (similaridad: 0.2391)
  4. 'homicides'  (similaridad: 0.2331)
  5. 'firearms'  (similaridad: 0.2328)

Palabra: 'medical'  →  Top 5 más similares:
  1. 'romano'  (similari


**Interpretación (Ejercicio 4):**

| Palabra | Top 5 más similares | Sim. máxima |
|---------|---------------------|:---:|
| `computer` | decwriter, deluged, harkens, shopper, the | 0.156 |
| `religion` | religious, religions, categorized, purpsoe, crusades | 0.245 |
| `space` | nasa, seds, shuttle, enfant, seti | 0.330 |
| `gun` | guns, crime, handgun, homicides, firearms | 0.358 |
| `medical` | romano, hospitals, recuperation, providers, relelvant | 0.282 |

`gun` y `space` tienen los vecinos más claros. Para `gun`: `guns`, `handgun`, `firearms`, `homicides`, `crime` son exactamente los términos que aparecen en `talk.politics.guns`. Para `space`: `nasa`, `shuttle`, `seti`, `seds` son todos de `sci.space`, con `enfant` como única excepción —probablemente algún post bilingüe que se coló en esa categoría.

`religion` recupera variantes morfológicas (`religious`, `religions`) y `crusades`, que tiene relación histórica obvia. `purpsoe` es un typo de "purpose" que aparece seguido en debates filosófico-religiosos. `categorized` es el único vecino que no dice mucho.

`medical` tiene dos vecinos claros (`hospitals`, `providers`) y uno temáticamente cercano (`recuperation`). Los otros dos (`romano`, `relelvant`) parecen ser typos frecuentes en los posts de `sci.med` que terminaron co-ocurriendo bastante con esa palabra.

El caso más interesante es `computer`. Sus vecinos tienen similaridades muy bajas (< 0.16) y son poco interpretables. La razón es que `computer` aparece en prácticamente todas las categorías `comp.*` y en `misc.forsale`, por lo que su vector en la matriz término-documento cubre un perfil de documentos muy heterogéneo. Cuando una palabra co-ocurre con vocabulario muy diverso, la similitud coseno con cualquier otra palabra cae porque no tiene un "vecindario" concentrado.

Esto marca una diferencia importante con modelos como Word2Vec: la similitud aquí mide co-ocurrencia a nivel de documento entero, no de contexto local. Palabras temáticamente específicas (`gun`, `space`) dan resultados mucho más útiles que palabras genéricas o transversales a todo el dataset (`computer`).
